<a href="https://colab.research.google.com/github/Iddrisu-Abdulai/Analysis_Python/blob/main/Book_Recommendation_Engine_using_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Book Recommendation System

This notebook builds a book recommendation system using the `GoodBooks-10k` dataset and `NearestNeighbors` algorithm from `scikit-learn`. The process involves data loading, preprocessing, model training, and generating recommendations.

In [4]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import os
import requests # Import requests for better download handling

# URLs for the GoodBooks-10k dataset (alternative to Book-Crossing due to 404 errors)
ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"

# Download ratings.csv
print(f"Downloading ratings from {ratings_url}...")
response = requests.get(ratings_url, stream=True)
response.raise_for_status()  # Raise an exception for bad responses (e.g., 404)
with open('ratings.csv', 'wb') as file:
    for chunk in response.iter_content(chunk_size=1024):
        if chunk:
            file.write(chunk)
print("Ratings downloaded.")

# Download books.csv
print(f"Downloading books from {books_url}...")
response = requests.get(books_url, stream=True)
response.raise_for_status()
with open('books.csv', 'wb') as file:
    for chunk in response.iter_content(chunk_size=1024):
        if chunk:
            file.write(chunk)
print("Books downloaded.")

# Load the datasets
books_raw = pd.read_csv('books.csv')
ratings_raw = pd.read_csv('ratings.csv')

# Select and rename columns for books to match original code's expectations
# GoodBooks-10k: 'book_id' as unique identifier, 'title' as book title, 'authors' as author
books = books_raw[['book_id', 'title', 'authors']]
books = books.rename(columns={'book_id': 'ISBN', 'title': 'Book-Title', 'authors': 'Book-Author'})

# Select and rename columns for ratings to match original code's expectations
# GoodBooks-10k: 'user_id', 'book_id', 'rating'
ratings = ratings_raw[['user_id', 'book_id', 'rating']]
ratings = ratings.rename(columns={'user_id': 'User-ID', 'book_id': 'ISBN', 'rating': 'Book-Rating'})

# GoodBooks-10k ratings are typically 1-5, Book-Crossing were 0-10.
# The original code did not normalize ratings, so we proceed with the GoodBooks-10k scale.
# GoodBooks-10k CSVs are generally clean, so 'on_bad_lines'/'error_bad_lines' are not needed.

Ratings downloaded.
Books downloaded.


## 1. Data Loading and Preprocessing

We start by downloading and loading the `ratings.csv` and `books.csv` datasets from the `GoodBooks-10k` repository. We then rename relevant columns to maintain compatibility with the subsequent code, mapping `book_id` to `ISBN`, `title` to `Book-Title`, `authors` to `Book-Author`, `user_id` to `User-ID`, and `rating` to `Book-Rating`.

In [10]:
# Filter data for statistical significance
# Further reduced filtering thresholds to ensure data is retained and prevent empty matrix issues
user_ratings_count = ratings.groupby('User-ID')['Book-Rating'].count()
users_to_keep = user_ratings_count[user_ratings_count >= 1].index  # Reduced from 10 to 1
filtered_ratings = ratings[ratings['User-ID'].isin(users_to_keep)]

book_ratings_count = filtered_ratings.groupby('ISBN')['Book-Rating'].count()
books_to_keep = book_ratings_count[book_ratings_count >= 1].index  # Reduced from 10 to 1
filtered_ratings = filtered_ratings[filtered_ratings['ISBN'].isin(books_to_keep)]

## 2. Data Filtering for Significance

To ensure the model learns from meaningful data, we filter out users and books with too few ratings. This step helps in reducing noise and improving the quality of recommendations. The thresholds have been set to `1` to retain as much data as possible given the dataset characteristics.

In [9]:
# Create a pivot table of user ratings
# Using pivot_table to handle potential duplicate (ISBN, User-ID) pairs gracefully
# with a mean aggregation function, which is suitable for ratings.
ratings_pivot = filtered_ratings.pivot_table(index='ISBN', columns='User-ID', values='Book-Rating', aggfunc='mean').fillna(0)

# Convert the pivot table to a sparse matrix
ratings_matrix = csr_matrix(ratings_pivot.values)

## 3. Creating the Rating Matrix

We create a pivot table (`ratings_pivot`) where rows represent books (by ISBN), columns represent users (by User-ID), and values are the book ratings. Missing ratings are filled with `0`. This pivot table is then converted into a sparse matrix (`ratings_matrix`), which is an efficient way to store data with many zeros, suitable for the `NearestNeighbors` model.

In [11]:
# Create and fit the KNN model
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')

# Defensive check: Ensure ratings_matrix is not empty before fitting
if ratings_matrix.shape[0] == 0 or ratings_matrix.shape[1] == 0:
    print("Error: ratings_matrix is empty. Please ensure the data filtering and pivot table creation steps (cells _szOlJaq7GHD and My8iT2DM7MEL) have successfully generated a non-empty ratings matrix.")
else:
    model_knn.fit(ratings_matrix)
    print("KNN model fitted successfully.")

KNN model fitted successfully.


## 4. Training the K-Nearest Neighbors Model

A `NearestNeighbors` model is initialized with `cosine` similarity as the metric and `brute` force algorithm (due to the sparse nature and potential size of the matrix). The model is then fitted to the `ratings_matrix` to find the nearest neighbors based on rating patterns.

In [12]:
# Function to get recommendations
def get_recommends(book_title):
    book_index = books[books['Book-Title'] == book_title].index[0]
    isbn = books.iloc[book_index]['ISBN']
    distances, indices = model_knn.kneighbors(ratings_pivot.loc[isbn].values.reshape(1, -1), n_neighbors=6)
    recommended_books = []
    for i in range(1, len(distances.flatten())):
        recommended_books.append({
            'title': books.iloc[ratings_pivot.index.get_loc(ratings_pivot.index[indices.flatten()[i]])]['Book-Title'],
            'distance': distances.flatten()[i]
        })
    return recommended_books

## 5. Recommendation Function

The `get_recommends` function takes a `book_title` as input. It finds the corresponding book in the `books` DataFrame, retrieves its rating vector from the `ratings_pivot` table, and then uses the trained `model_knn` to find the 5 nearest books (excluding the input book itself) based on rating similarity. The function returns a list of recommended book titles and their distances.

In [14]:
recommendations = get_recommends("The Hunger Games (The Hunger Games, #1)")
print(recommendations)

[{'title': 'Catching Fire (The Hunger Games, #2)', 'distance': np.float64(0.27698877880282213)}, {'title': 'Mockingjay (The Hunger Games, #3)', 'distance': np.float64(0.3126530413102985)}, {'title': "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", 'distance': np.float64(0.4101597031858163)}, {'title': 'Twilight (Twilight, #1)', 'distance': np.float64(0.4388581040880518)}, {'title': 'Divergent (Divergent, #1)', 'distance': np.float64(0.47668349347347805)}]


## 6. Example Usage

Here's an example of how to use the `get_recommends` function to find similar books for "The Hunger Games (The Hunger Games, #1)". The output shows the recommended books along with their cosine distances, where a lower distance indicates higher similarity.